In [1]:
import sys, asyncio

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [2]:

import re
from typing import List, Dict, Any
from playwright.async_api import async_playwright

In [3]:
async def extract_html_table(table) -> List[Dict[str, str]]:
    # only visible/non-hidden headers
    header_locs = table.locator("thead tr th:not(.ocultar)")
    headers = [h.strip() for h in await header_locs.all_inner_texts() if h.strip()]

    rows = []
    tr_loc = table.locator("tbody tr")
    tr_count = await tr_loc.count()

    for i in range(tr_count):
        td_locs = tr_loc.nth(i).locator("td:not(.ocultar)")
        values = [v.strip() for v in await td_locs.all_inner_texts()]
        row = {h: (values[idx] if idx < len(values) else "") for idx, h in enumerate(headers)}
        if len(values) > len(headers):
            row["_extra_values"] = values[len(headers):]
        rows.append(row)

    return rows


In [4]:
async def extract_main_table(page) -> List[Dict[str, str]]:
    main_table = page.locator("table[role='table']:has(td.p-link2)").first
    await main_table.wait_for(state="visible", timeout=60_000)
    return await extract_html_table(main_table)


In [5]:
async def open_dialog_and_extract_table(page) -> List[Dict[str, str]]:
    link = page.locator("td.p-link2").first
    if await link.count() == 0:
        return []

    await link.click()

    dialog = page.locator("div[role='dialog']").first
    await dialog.wait_for(state="visible", timeout=60_000)

    dialog_table = dialog.locator("table[role='table']").first
    await dialog_table.wait_for(state="visible", timeout=60_000)

    data = await extract_html_table(dialog_table)

    # close dialog (optional)
    close_btn = dialog.locator("button.p-dialog-header-close, button:has-text('Cerrar')").first
    if await close_btn.count() > 0:
        await close_btn.click()

    return data


In [6]:
async def extract_both_tables(url: str, headless: bool = True) -> Dict[str, Any]:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        ctx = await browser.new_context(viewport={"width": 1400, "height": 900})
        page = await ctx.new_page()

        await page.goto(url, wait_until="domcontentloaded")
        await page.wait_for_timeout(1500)  # let Angular render

        main_rows = await extract_main_table(page)
        dialog_rows = await open_dialog_and_extract_table(page)

        await ctx.close()
        await browser.close()

    return {"url": url, "main_table": main_rows, "dialog_table": dialog_rows}


In [7]:
import pandas as pd
new_tenders = pd.read_excel('data/Tender List.xlsx', sheet_name='New Tenders')
URL_COL = 'DirecciÛn anuncio'

print(*new_tenders[URL_COL].head())

https://comprasmx.buengobierno.gob.mx/sitiopublico/#/sitiopublico/detalle/0c434b009d084925aa384c1a3374ec1e/procedimiento https://comprasmx.buengobierno.gob.mx/sitiopublico/#/sitiopublico/detalle/e4cabbca2b054400aa75c373c8f6b31b/procedimiento https://comprasmx.buengobierno.gob.mx/sitiopublico/#/sitiopublico/detalle/1e4d442993fd457cb3ca305e6ec2715e/procedimiento https://comprasmx.buengobierno.gob.mx/sitiopublico/#/sitiopublico/detalle/611c20db83e841c09b63796b70e9391c/procedimiento https://comprasmx.buengobierno.gob.mx/sitiopublico/#/sitiopublico/detalle/20a29614e6584632a08f9136235a54cc/procedimiento


In [8]:
sample_url = new_tenders[URL_COL].dropna().iloc[0]
result = await extract_both_tables(sample_url, headless=False)

len(result["main_table"]), len(result["dialog_table"])


NotImplementedError: 